## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 122.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 224.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 305.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 263.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-o

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub pandas tiktoken pymupdf langchain langchain-community langchain-text-splitters chromadb pip install sentence-transformers langchain-huggingface  numpy -q

ERROR: Could not find a version that satisfies the requirement install (from versions: none)
ERROR: No matching distribution found for install


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [3]:
# Defining the Hugging Face repository and model version for Mistral-7B
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
# Specifying the file name for the quantized Mistral-7B model
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [4]:
# Downloading the specified model file from Hugging Face Hub and store its local path
model_path = hf_hub_download(
    repo_id= model_name_or_path, #Complete the code to mention the repo id
    filename= model_basename #Complete the code to mention the model name
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Loading the LLaMA model with specified context, GPU layers, and batch size
llm = Llama(
    model_path=model_path,
    n_ctx=5300,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 1 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [6]:
def response(query,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
   # Sends the query to the LLM with specified parameters
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    # Extracting and returning only the text part of the response
    return model_output['choices'][0]['text']

In [7]:
response("What treatment options are available for managing stomach pain?")

"\n\nStomach pain, also known as abdominal pain, can have various causes and may require different treatments. Here are some common treatment options for managing stomach pain:\n\n1. Medications: Over-the-counter (OTC) or prescription medications may be recommended to help alleviate stomach pain. These include antacids for heartburn and acid reflux, anti-diarrheal agents for diarrhea, antibiotics for bacterial infections, and pain relievers such as acetaminophen or nonsteroidal anti-inflammatory drugs (NSAIDs) for inflammation or pain.\n2. Dietary modifications: Making dietary changes can help manage stomach pain caused by conditions like irritable bowel syndrome (IBS), lactose intolerance, or food allergies. Avoiding trigger foods and eating smaller, more frequent meals may help reduce symptoms.\n3. Probiotics: Probiotics are live bacteria and yeasts that can help restore the balance of good bacteria in the gut, which can alleviate stomach pain caused by conditions like IBS or antibio

####Observations


*   The response generated by the LLM is accurate and medically appropriate for managing stomach pain
*   Trucated output suggests token limitation restricts complete responses



### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [8]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clin

####Observations


*   The response generated by the LLM is accurate and medically appropriate for managing sepsis in critical care unit

*   Trucated output suggests token limitation restricts complete responses





### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [9]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


"\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.\n3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.\n4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.\n5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.\n6. Abdominal swelling: The abdomen may become swollen and tender to the touch.\n7. Inability to pass gas: Passing gas can be difficult due to the infla

####Observations


*   The response generated by the LLM is accurate and medically appropriate. It accurately identifies the symptoms for appendicities and treatment options
*   Trucated output suggests token limitation restricts complete responses


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [10]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.\n\nThere are several treatments that have been shown to be effective in addressing sudden patchy hair loss:\n\n1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.\n2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increa

####Observations


*   The response generated by the LLM is accurate and medically appropriate. It accurately identifies the condition and causes behind it
*   Trucated output suggests token limitation restricts complete responses

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [11]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a brain injury, such as pain, swelling, seizures, or infections.\n3. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.\n4. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. It may include physical therapy, occupational therapy, speech and langua

####Observations


*   The response generated by the LLM is accurate and medically appropriate. It accurately identifies the treatment needed
*   Trucated output suggests token limitation restricts complete responses

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [12]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.\n4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.\n5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, seek medical help as soon as possible.\n\nOnce you've ensured the person's safety and stabi

####Observations


*   The response generated by the LLM is accurate but more generic  
*   Trucated output suggests token limitation restricts complete responses

####Overall Observations for LLMs


Strengths
  

*   Accurate Responses - The Model provides detailed and medically appropriate answers covering broad information domain
*   Structure - The Model responds in clear structured format

Limitations


*   Generic responses - While the answers were appropriate, the responses were generic.
*   Incomplete responses - LLM token limit caused the responses to be truncated. Need to experiment with different limits to find the appropriate token limits
*   No citations or evidence grounding. Potential hallucination risk in unseen domains





## Question Answering using LLM with Prompt Engineering

In [13]:
#system prompt describing the assistant's role.
system_prompt = """
You are an AI Assistant who has expertise in medical domain. You are expected to be clear to the point and provide relevant response based on the context.
Your responses should be accepted by medical professional or medical standards.
If you do not know the answer, please say "I don't know" and avoid giving any generic response.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [14]:
input1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(input1)

Llama.generate: prefix-match hit


'\n\nThe protocol for managing sepsis in a critical care unit involves early recognition, rapid resuscitation, source control, antibiotic therapy, and close monitoring. Here are the steps:\n\n1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, tachycardia, hypotension), and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.\n\n2. Rapid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore circulating volume. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider using vasopressors if necessary to maintain adequate blood pressure.\n\n3. Source Control: Identify and address the source of infection, which may include surgical intervention or drainage procedures.\n\n4. Antibiotic Therapy: Administer broad-spectrum antibiotics based on suspected pathogens 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
input2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(input2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:\n\n1. Sudden pain in the lower right abdomen, which may be mild at first but becomes sharp and severe over time.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, which may start after the abdominal pain begins.\n5. Constipation or diarrhea.\n6. Abdominal swelling and tenderness.\n7. Inability to pass gas or have a bowel movement.\n8. Feeling restless or unable to find a comfortable position due to pain.\n\nAppendicitis cannot be cured via medicine alone, as the inflammation can lead to the appendix rupturing if left untreated. If the appendix ruptures, it can lead to peritonitis, a serious infection of the abdominal cavity that requires immediate surgical intervention. The standard treatment for appendicitis is an appendectomy, which i

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
input3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(input3)

Llama.generate: prefix-match hit


"\n\nPossible causes of sudden patchy hair loss, also known as alopecia areata, include:\n1. Autoimmune disorders: In this condition, the body's immune system attacks the hair follicles, leading to hair loss.\n2. Stress: Emotional or physical stress can trigger hair loss.\n3. Vitamin deficiencies: Lack of certain vitamins like iron, zinc, and biotin can cause hair loss.\n4. Hormonal imbalances: Thyroid disorders, polycystic ovary syndrome (PCOS), and other hormonal imbalances can lead to hair loss.\n5. Medications: Certain medications like chemotherapy drugs, antidepressants, and beta-blockers can cause hair loss as a side effect.\n6. Infections: Fungal infections like ringworm or bacterial infections can cause patchy hair loss.\n7. Trauma: Physical trauma to the scalp, such as burns or injuries, can lead to hair loss.\n\nEffective treatments for addressing sudden patchy hair loss include:\n1. Corticosteroids: Topical or injected corticosteroids can help reduce inflammation and promote

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
input4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(input4)

Llama.generate: prefix-match hit


"\n\nA person with a brain injury may require a combination of medical interventions and rehabilitative therapies depending on the severity and location of the injury. Here are some common treatments:\n\n1. Emergency care: For severe brain injuries, immediate medical attention is necessary to prevent further damage or complications. This may include surgery to remove hematomas or decompressing skull fractures, administering medications to control swelling or seizures, and providing supportive care such as maintaining breathing and blood pressure.\n2. Medications: Depending on the symptoms, various medications may be prescribed to manage conditions such as pain, seizures, infections, and depression. For instance, anti-inflammatory drugs can help reduce swelling, while anticonvulsants are used to prevent seizures.\n3. Rehabilitative therapies: These therapies aim to help the person regain lost skills or compensate for impairments. Some common rehabilitative therapies include physical the

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [18]:
input5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(input5)

Llama.generate: prefix-match hit


"\n\nA fractured leg, specifically a tibia or fibula fracture, is a common injury that can occur during hiking due to falls, tripping over rocks, or other traumatic events. Here are the necessary precautions and treatment steps for such an individual:\n\n1. Assess the severity of the injury: Check for signs of open wounds, deformities, swelling, numbness, or inability to move the leg. If there is significant bleeding, apply a clean cloth to control it and elevate the leg to reduce swelling. Do not attempt to realign the bone if it is visibly misaligned.\n2. Immobilize the leg: Use a splint, sling, or a makeshift immobilizer made from available materials (such as sticks, clothing, or backpack) to prevent further movement and potential damage to the fracture. Be sure to keep the injured leg elevated above heart level to minimize swelling and pain.\n3. Seek medical attention: It is essential to seek professional medical help as soon as possible. Fractures can lead to complications such as

####Overall Observations for LLMs with Prompt Engineering
Strengths

*   A clearly defined system prompt helps with the tone, accuracy and structured reasoning
*   Providing anti-hallucination instructions enhances safety and trustworthiness
*   Demonstrates contextual medical reasoning in responses

Limitations

*  Generic responses - While the answers were appropriate, the responses were generic.
*  Incomplete responses - LLM token limit caused the responses to be truncated. Need to experiment with different limits to find the appropriate token limits
No citations or evidence grounding. Potential hallucination risk in unseen domains

## Question Answering using LLM with Prompt Engineering + Parameter Tuning

### Impact of temperature

In [19]:
input1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
response(input1, temperature=0.5)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition caused by a dysregulated host response to infection. In a critical care unit, managing sepsis involves addressing both the underlying infection and the systemic inflammatory response. Here are some general steps for managing sepsis in a critical care unit:\n\n1. Early recognition and diagnosis: Recognize sepsis early based on clinical signs and symptoms, such as fever or hypothermia, tachycardia or bradycardia, altered mental status, and lactic acidosis. Use scoring systems like Sequential Organ Failure Assessment (SOFA) or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) to help identify patients at risk for sepsis.\n2. Source control: Identify and address the source of infection as soon as possible. This may involve surgical intervention, such as drainage of an abscess or debridement of necrotic tissue.\n3. Fluid resuscitation: Aggressively replace intravascular volume loss with crystalloid fluids to maintain adequate tiss

####Observations


*   The response generated by the LLM is still coherent and factual but a bit more natural

### Impact of Top_p(0.95->0.7)

In [20]:

response(input1, top_p=0.7)

Llama.generate: prefix-match hit


'\n\nThe protocol for managing sepsis in a critical care unit involves early recognition, rapid resuscitation, source control, antibiotic therapy, and supportive measures. Here are the steps:\n\n1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, tachycardia, hypotension), and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.\n\n2. Rapid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore circulating volume. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider using vasopressors if necessary to maintain adequate blood pressure.\n\n3. Source Control: Identify and address the source of infection, which may include surgical intervention or drainage procedures.\n\n4. Antibiotic Therapy: Administer broad-spectrum antibiotics based on suspected pathoge

####Observations


*   The response generated by the LLM is more predictable with only the most confident choices selected

### Impact of Top_k(50->10)

In [21]:
response(input1, top_k=10)

Llama.generate: prefix-match hit


'\n\nThe protocol for managing sepsis in a critical care unit involves early recognition, rapid resuscitation, source control, antibiotic therapy, and supportive measures. Here are the steps:\n\n1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, tachycardia, hypotension), and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.\n\n2. Rapid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore circulating volume. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider using vasopressors if necessary to maintain adequate blood pressure.\n\n3. Source Control: Identify and address the source of infection, which may include surgical intervention or drainage procedures.\n\n4. Antibiotic Therapy: Administer broad-spectrum antibiotics based on suspected pathoge

####Observations


*   The response generated by the LLM is more deterministic, focused and factual

### Impact of Max Tokens(512->1024)

In [22]:
response(input1,max_tokens=1024)

Llama.generate: prefix-match hit


'\n\nThe protocol for managing sepsis in a critical care unit involves early recognition, rapid resuscitation, source control, antibiotic therapy, and supportive measures. Here are the steps:\n\n1. Early Recognition: Identify sepsis suspects based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidosis, tachycardia, hypotension), and organ dysfunction using Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score.\n\n2. Rapid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore circulating volume. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider using vasopressors if necessary to maintain adequate blood pressure.\n\n3. Source Control: Identify and address the source of infection, which may include surgical intervention or drainage procedures.\n\n4. Antibiotic Therapy: Administer broad-spectrum antibiotics based on suspected pathoge

Observations


*   The response generated is the same as before, additional tokens make no difference




### Impact of Temperate(0.3) and Top_p(0.8)

In [23]:
response(input1, temperature=0.3,top_p=0.8)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and management in a critical care unit. The following are the general steps for managing sepsis in a critical care unit:\n\n1. Early Recognition: Look for signs and symptoms of sepsis such as fever, chills, tachycardia, respiratory distress, altered mental status, and lactic acidosis. Use validated scoring systems like Sequential Organ Failure Assessment (SOFA) or Quick Sequential Organ Failure Assessment (qSOFA) to identify patients at risk of sepsis.\n2. Fluid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and prevent hypotension. Aim for a mean arterial pressure (MAP) of 65 mmHg or higher, but avoid over-resuscitation that can lead to fluid overload.\n3. Antibiotics: Start broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Adjust antibiotic therapy based on culture resu

Observations

*   The response is relevant, focused and factual. A good sweet spot after fine tuning

## Data Preparation for RAG

In [24]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

### Loading the Data

In [25]:
from google.colab import drive
drive.mount('/content/drive')
manual_pdf_path = "/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Data Overview

#### Checking the first 5 pages

In [26]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
uday.desiraju@gmail.com
X4SYCGN7EW
This file is meant for personal use by uday.desiraju@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
uday.desiraju@gmail.com
X4SYCGN7EW
This file is meant for personal use by uday.desiraju@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................................

Observations

*   Data is loaded correctly

#### Checking the number of pages

In [27]:
len(manual)

4114

Observations

*   Overall page length is 4114

### Data Chunking

In [28]:
# Initializing a RecursiveCharacterTextSplitter to split the text into manageable chunks for embedding and retrieval
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, #Complete the code to define the chunk size
    chunk_overlap= 20 #Complete the code to define the chunk overlap
)

Observations
* Data is split into chunk size of 512 with overlap size 20 (to maintain the context)

In [29]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [30]:
len(document_chunks)

8469

In [31]:
document_chunks[0].page_content

'uday.desiraju@gmail.com\nX4SYCGN7EW\nThis file is meant for personal use by uday.desiraju@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [32]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

In [33]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nuday.desiraju@gmail.com\nX4SYCGN7EW\nThis file is meant for personal use by uday.desiraju@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

### Embedding

In [34]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

/tmp/ipykernel_13369/3102610685.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [35]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [38]:
#Checking if both are of the same size
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [39]:
embedding_1,embedding_2

([-0.01111602783203125,
  -0.00788116455078125,
  0.003940582275390625,
  -0.01641845703125,
  -0.015167236328125,
  -0.029083251953125,
  -0.0189971923828125,
  0.038116455078125,
  0.018798828125,
  0.041290283203125,
  0.025543212890625,
  0.001903533935546875,
  0.027618408203125,
  -0.0244903564453125,
  -0.00473785400390625,
  -0.0038700103759765625,
  0.00313568115234375,
  -0.0241851806640625,
  0.0011034011840820312,
  0.0122833251953125,
  0.01165771484375,
  0.018890380859375,
  -0.06524658203125,
  -0.0391845703125,
  0.005687713623046875,
  0.037689208984375,
  -0.0003285408020019531,
  -0.02203369140625,
  0.05419921875,
  0.06146240234375,
  -0.0183868408203125,
  -0.005184173583984375,
  0.0214080810546875,
  -0.04022216796875,
  -0.022705078125,
  -0.00759124755859375,
  0.057769775390625,
  -0.03265380859375,
  -0.023406982421875,
  -0.042266845703125,
  0.00937652587890625,
  -0.0230865478515625,
  0.047454833984375,
  -0.016082763671875,
  -0.049041748046875,
  -0.0

### Vector Database

In [41]:
# Creating the output directory 'medicalinfo_db' if it doesn't already exist, so we can save the processed data or vector database files there.

out_dir = 'medicalinfo_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [42]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [43]:
#Loading Chroma vector store with the given embedding model
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_13369/2308006168.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [44]:
#Accessing the embedding function used in the Chroma vector store
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [45]:
#Performing a similarity search in the vector store to find the top 2 most similar documents
vectorstore.similarity_search("how to treat covid?",k=2)

[Document(metadata={'keywords': '', 'author': '', 'total_pages': 4114, 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'page': 1573, 'moddate': '2026-04-25T02:36:53+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'trapped': '', 'modDate': 'D:20260425023653Z'}, page_content='antiviral treatment in these patients appears to reduce the incidence of lower respiratory disease and\nhospitalization. Appropriate antibacterial therapy decreases the mortality rate due to secondary bacterial\npneumonia.\nTreatment\n• Symptomatic treatment\n• Sometimes antiviral drugs\nTreatment for most patients is symptomatic, including rest, hydration, and antipyretics

Observations
* From the retrieved chunks, we observe that all the chunks are related to treating covid

### Retriever

In [46]:
#Retrieving the top 3 most similar documents for a given query
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [47]:
rel_docs = retriever.invoke("What are treatments for common cold?")
rel_docs

[Document(metadata={'keywords': '', 'trapped': '', 'file_path': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'page': 1570, 'author': '', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'source': '/content/drive/MyDrive/UTAI/Data/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-04-25T02:36:53+00:00', 'modDate': 'D:20260425023653Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'format': 'PDF 1.7', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'total_pages': 4114}, page_content='ipratropium bromide (2 sprays of a 0.03% solution bid or tid); however, these drugs should be avoided in\nthe elderly and people with benign prostatic hypertrophy or glaucoma. First-generation antihistamines\nfrequently cause sedation, but 2nd-generation (nonsedating) antihistamines are ineffective for treating the\ncommon cold. Antihistamines and decongestants are not reco

### System and User Prompt Template

In [48]:
# System message instructing the LLM to only answer using Merck Manual
qna_system_message = """
You are a medical knowledge assistant designed to support healthcare professionals by providing accurate, reliable, and concise medical information.

Your task is to answer user questions using ONLY the information provided in the context, which is sourced from trusted medical references such as the Merck Manuals.

The context will begin with the token: ###Context
The user question will begin with the token: ###Question

Rules you must follow:
- Use only the information explicitly stated in the provided context.
- Do NOT add external medical knowledge, assumptions, or personal opinions.
- Do NOT mention or refer to the context, documents, or source materials in your final answer.
- If the answer is not available or cannot be clearly derived from the context, respond exactly with:
  "I don't know"

Your responses should be:
- Clinically accurate
- Clear and concise
- Structured where appropriate (bullet points or steps)
- Written in professional medical language suitable for healthcare practitioners
"""

In [49]:
qna_user_message_template = """
###Context
The following excerpts are retrieved from medical reference documents relevant to the question below.

{context}

###Question
{question}
"""

### Response Function

In [50]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(input=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?





In [51]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,top_k=20)

Llama.generate: prefix-match hit


"Based on the provided context, here's the answer:\n\n- Monitor vital signs and quantify all fluid intake and output.\n- Conduct daily electrolytes and CBC tests.\n- For patients with arrhythmias, measure Mg, phosphate, and Ca levels.\n- For patients receiving TPN, perform weekly liver enzymes and coagulation profiles.\n- Suspect sepsis and obtain blood cultures for Gram stain and culture before administering empiric antibiotics.\n- Start immediate empiric antibiotic therapy based on suspected source, clinical setting,"

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [52]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided:\n\nSymptoms of appendicitis include abdominal pain, anorexia (loss of appetite), and abdominal tenderness. There is no mention of a cure through medication in the context. The treatment for appendicitis is surgical removal, specifically open or laparoscopic appendectomy.'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [53]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided:\n\n1. Alopecia Areata: This is a common cause of sudden patchy hair loss. It is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Treatment options include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).\n2. Traction alo'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [54]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided:\n- The initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.\n- In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n- There is no specific treatment for tra'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [55]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_k=20)

Llama.generate: prefix-match hit


"Based on the provided context, here's the answer:\n\n1. Immobilize the injured limb using elastic bandages or a splint to prevent further injury and facilitate healing. Elevate the limb above heart level for the first 2 days to minimize swelling.\n2. Apply warmth (heating pad) for 15 to 20 minutes after the initial 48 hours to relieve pain and speed up healing.\n3. Keep the cast or splint dry, never put any object inside it, inspect the edges and skin around it daily, and apply lotion to"

RAG System Observations
* Strong semantic retrieval for the queries
* Some of the responses were truncated due to token limit
* May need hypertuning vector configuration to improve performance
* Lesser hallucination risk


### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?




In [56]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_1,temperature=0.5)

Llama.generate: prefix-match hit


'Based on the provided context, the following is the answer:\n\n1. Initial empiric antibiotic therapy:\n   - Regimen 1: Gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day).\n   - If Pseudomonas is suspected: Ceftazidime 2 g IV q 8 h.\n   - Alternatively,'

Observations
* No observed increase in creativity
* Clinical correctness remains the same, no hallucinations
* Response truncated due to token limit



###Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


In [57]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,temperature=0.5, max_tokens =128)

Llama.generate: prefix-match hit


'###Answer\nCommon symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; it requires surgical removal, specifically an open or laparoscopic appendectomy.'

Observations
* No observed increase in creativity
* Clinical correctness remains the same, no hallucinations
* Response not truncated due to token limit

###Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [58]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_p=0.9, top_k=30)

Llama.generate: prefix-match hit


'Based on the context provided:\n\n1. Alopecia Areata: This is a common cause of sudden patchy hair loss. It is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Treatment options include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).\n2. Traction alo'

Observations
* More deterministic and focused medical reasoning
* Clinical correctness remains the same, no hallucinations
* Response truncated due to token limit

###Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [59]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_p=0.9, top_k=10)

Llama.generate: prefix-match hit


'Based on the context provided:\n- The initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.\n- In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.\n- There is no specific treatment for tra'

Observations
* More deterministic and focused medical reasoning
* Improves clinical reliability and consistency
* Response truncated due to token limit

###Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [63]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_p= 0.9, top_k=10,max_tokens=512)

Llama.generate: prefix-match hit


"Based on the provided context, here's the answer:\n\n1. Initial Care:\n   - Elevate the injured limb above heart level for first 2 days to minimize swelling.\n   - Apply ice intermittently after 48 hours to relieve pain and speed healing.\n   - Immobilize the injury using a cast or splint, depending on the severity and duration of immobilization required.\n   - Keep the cast dry and inspect the skin around it daily for redness or soreness.\n   - Avoid putting objects inside the cast.\n   - Seek medical care if an odor emanates from within the cast or a fever develops, which may indicate infection.\n\n2. Immobilization:\n   - Immobilization decreases pain and facilitates healing by preventing further injury.\n   - Joints proximal and distal to the injury should be immobilized as well.\n   - A cast is typically used for fractures or other injuries that require weeks of immobilization.\n   - Prolonged immobilization (more than 3-4 weeks) can cause complications such as stiffness, contrac

Observations
* More deterministic and focused medical reasoning
* Improves clinical reliability and consistency
* Response still truncated due to token limit but better than lower max tokens

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [64]:
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [65]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [66]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [67]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(input=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [68]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input_1,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare the AI generated answer with the identified key information.
3. Determine if the answer is derived only from the information presented in the context.

Explanation:
The AI generated answer includes the following steps for managing sepsis in a critical care unit: prompt empiric antibiotic therapy, drainage of abscesses and surgical excision of necrotic tissues, normalization of blood glucose, corticosteroid therapy, consideration of activated protein C, monitoring and testing, and continuation of antibiotics.

The context provides information on the importance of ICU care for critically ill patients, patient monitoring and testing, supportive care including prevention of infection, and specific information on managing sepsis such as prompt empiric therapy with antibiotics, drainage of abscesses, normalization of blood glucose, corticosteroid ther

Observations
* Groundness rating - 5 - Answer if fully derived from the provided context
* Relevance rating - 4 - Answer addresses most of aspects of the question while also excluding irrelevant information

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [69]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input_2,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis and its symptoms.
2. Determine if the AI generated answer includes only the information derived from the context.
3. Evaluate the extent to which the metric is followed.

Explanation:
The context provides information about appendicitis, including its common symptoms (abdominal pain, anorexia, and abdominal tenderness). The AI generated answer also mentions these same symptoms for appendicitis. Additionally, the answer states that appendicitis cannot be cured via medicine alone and that the standard treatment is surgical removal of the appendix through open or laparoscopic appendectomy. This information is directly from the context. Therefore, the AI generated answer adheres to the metric as it is derived only from the information presented in the context.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the answer a 5 (The metric is fo

Observations

* Groundness rating - 5 - Answer if fully derived from the provided context
* Relevance rating - 5 - Answer addresses all aspects of the question while also excluding irrelevant information

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [70]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input_3,max_tokens=350)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the main topic of the question and context. In this case, it's about effective treatments and possible causes for sudden patchy hair loss.
2. Check if the AI generated answer is derived only from the information presented in the context.
3. If yes, rate the extent to which the metric is followed based on the evaluation criteria (1-5).
4. If no, provide a step-by-step explanation of why the answer does not adhere to the metric and assign a score accordingly.

Evaluation:
The AI generated answer adheres to the metric as it mentions various treatments for sudden patchy hair loss that are explicitly stated in the context (topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy, hormonal modulators, and surgical options). The possible causes mentioned in the answer (autoimmune disorders, stress, or certain medications) are also consistent with the context. Therefore, the metric is followed mo

Observations

* Groundness rating - 4 - Answer is mostly derived from the provided context
* Relevance rating - 4 - Answer addresses most of aspects of the question while also excluding irrelevant information

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [71]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input_4,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to treatments for traumatic brain injury (TBI).
2. Compare the AI generated answer with the identified information from the context.
3. Determine if the answer is derived only from the information presented in the context.

Explanation:
The AI generated answer includes all the treatments mentioned in the context, such as ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure; surgery to place monitors or decompress the brain; supportive care including preventing systemic complications, providing good nutrition, and preventing pressure ulcers; and long-term treatment with drugs for intractable seizures. The answer also mentions the importance of maintaining adequate brain perfusion and oxygenation in the first few days after injury and rehabilitation for many patients.

The information in the context is extensive and covers various aspects of TBI, including diagnos

Observations

* Groundness rating - 5 - Answer if fully derived from the provided context
* Relevance rating - 4 - Answer addresses most of aspects of the question while also excluding irrelevant information

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [72]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input_5,max_tokens=400)
print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information directly related to the question in the context.
2. Check if the AI generated answer only includes information derived from the identified context.
3. Compare each point in the AI generated answer with the corresponding information in the context.

The answer adheres to the metric considering the question and context as input:
Yes, the AI generated answer is based solely on the information provided in the context. Each step mentioned in the answer can be traced back to specific details within the context.

Extent to which the metric is followed:
5 - The metric is followed completely.

Rating and explanation:
Based on the evaluation criteria, I would rate the AI generated answer as a 5 because it follows the metric completely by deriving all information from the context provided.

 Steps to evaluate context as per relevance metric:
1. Identify the main aspects of the question: necessary precautions and treatment steps for a pers

Observations

* Groundness rating - 5 - Answer if fully derived from the provided context
* Relevance rating - 5 - Answer addresses all aspects of the question(initial care, immobilization techniques, instructions for patients with casts, pain relief methods) while also excluding irrelevant information

## Actionable Insights and Business Recommendations

###Business Insights
* The RAG based system shows strong potential to aid medical diagnosis and accelerate clinical decision-making
* Accuracy and Trust - RAG-based answers are context-specific, reliable, and trustworthy thereby enhancing customer satisfaction in medical applications
* User experience - While not eliminating the need for experts, the system can act as a "first-pass" diagnostic aid. This reduces the time clinicians spend on manual literature reviews, directly accelerating the speed of informed decision-making
* Because the system prioritizes citation-heavy outputs, it creates an audit trail for clinical suggestions. This transparency is critical for navigating the regulatory and safety requirements of modern healthcare digital transformation
###Recommendations
* Position the tool as a "Co-Pilot." It should generate the evidence-based draft, but the final "Sign-Off" must always remain a distinct, manual action by the licensed professional
* Standard AI metrics don't measure safety. Since health is highly regulated, integration with medical frameworks to help avoid bias and enable trust may be needed before stand alone adoption
* May need to extend to specialized models tailored to specialties like pediatrics, oncology etc
* Need to define process to regularly update and finetune the model as new medical information becomes available.

<font size=6 color='blue'>Power Ahead</font>
___